In [ ]:
# Run this cell to install DiffeRT and its dependencies, e.g., on Google Colab

try:
    import differt  # ruff: ignore[unused-import]
except ImportError:
    import sys  # ruff: ignore[unused-import]

    !{sys.executable} -m pip install differt[all]

# EM Fields' ABC: Fundamentals of Electromagnetic Interactions

In radio propagation and ray tracing, high-frequency electromagnetic fields
interact with the physical environment through several distinct physical
mechanisms:

1. **Specular Reflection**: Plane surfaces reflect rays according to Snell's law,
   with reflection coefficients given by Fresnel formulas.
2. **Edge Diffraction**: Sharp edges scatter energy into geometric shadow zones
   governed by the Uniform Theory of Diffraction (UTD).
3. **Transmission**: Rays passing through finite-thickness dielectric slabs
   experience attenuation and phase delay.
4. **Diffuse Scattering**: Surface roughness scatters power into a broad angular
   lobe (e.g., Lambertian scattering).

This tutorial demonstrates each interaction mathematically and visually using DiffeRT.

In [ ]:
import differt.plotting as dplt
import jax.numpy as jnp
import matplotlib.pyplot as plt
from differt.em import (
    InteractionType,
    Material,
    compute_received_fields,
    compute_received_power,
    materials,
)
from differt.geometry import Mesh, Scene, TracedPaths
from jaxtyping import ArrayLike, Float

dplt.set_backend("plotly")

## Scene Geometry Setup

To examine these interactions in a controlled setting, we construct a 90-degree
wedge made of two concrete plates ($30\,\text{m} \times 30\,\text{m}$) sharing an edge along
the z-axis.

In [ ]:
frequency = 1e9  # 1 GHz carrier

# We define a 90-degree right-angle wedge made of concrete
wedge = (
    Mesh(
        vertices=jnp.array([
            [0.0, 0.0, -10.0],
            [0.0, 0.0, 10.0],
            [10.0, 0.0, -10.0],
            [10.0, 0.0, 10.0],
            [0.0, 10.0, -10.0],
            [0.0, 10.0, 10.0],
        ]),
        triangles=jnp.array([[0, 2, 1], [2, 3, 1], [0, 1, 4], [1, 5, 4]]),
    )
    .set_face_materials(0)
    .set_materials("itu_concrete")
)


def single_bounce_paths(
    tx: Float[ArrayLike, "3"],
    bounce_point: Float[ArrayLike, "... 3"],
    rx: Float[ArrayLike, "... 3"],
    object_index: int,
    interaction_type: InteractionType,
) -> TracedPaths:
    """Construct a synthetic single-bounce TracedPaths object."""
    tx_arr = jnp.broadcast_to(tx, (*rx.shape[:-1], 3))
    bp_arr = jnp.broadcast_to(bounce_point, (*rx.shape[:-1], 3))
    vertices = jnp.stack([tx_arr, bp_arr, rx], axis=-2)
    objects = jnp.broadcast_to(jnp.asarray([object_index]), (*rx.shape[:-1], 1))
    itypes = jnp.broadcast_to(
        jnp.asarray([int(interaction_type)]), (*rx.shape[:-1], 1)
    )
    mask = jnp.ones((*rx.shape[:-1], 1), dtype=bool)
    return TracedPaths(
        vertices=vertices,
        objects=objects,
        interaction_types=itypes,
        mask=mask,
    )

## 1. Specular Reflection: Two-Ray Interference

When both a direct Line-of-Sight (LoS) path and a ground-reflected path reach
the receiver, their complex electric fields superimpose:

$$E_{\text{total}} = E_{\text{LoS}} + E_{\text{refl}}$$

Because the path lengths differ, the phase difference $\Delta \phi = \frac{2\pi}{\lambda} \Delta d$
cycles periodically as the receiver moves, creating characteristic constructive and
destructive interference fringes (two-ray multipath fading).

In [ ]:
tx_refl = jnp.array([-1.0, -25.0, 0.0])
d_rx = jnp.linspace(0.5, 15.0, 500)
rx_refl = jnp.stack(
    [-d_rx, -25.0 + 20.0 * (d_rx / 15.0), jnp.zeros_like(d_rx)], axis=-1
)
bounce_refl = jnp.stack(
    [
        jnp.zeros_like(d_rx),
        -25.0 + 20.0 * (1.0 / (1.0 + d_rx)),
        jnp.zeros_like(d_rx),
    ],
    axis=-1,
)

refl_paths = single_bounce_paths(
    tx=tx_refl,
    bounce_point=bounce_refl,
    rx=rx_refl,
    object_index=0,
    interaction_type=InteractionType.REFLECTION,
)
los_paths = TracedPaths(
    vertices=jnp.stack(
        [jnp.broadcast_to(tx_refl, rx_refl.shape), rx_refl], axis=-2
    ),
    objects=-jnp.ones((*d_rx.shape, 2), dtype=int),
    mask=jnp.ones_like(d_rx, dtype=bool),
    interaction_types=jnp.zeros((*d_rx.shape, 0), dtype=int),
)

refl_field = compute_received_fields(refl_paths, wedge, frequency)
los_field = compute_received_fields(los_paths, wedge, frequency)
total_field = refl_field + los_field

plt.figure(figsize=(8, 4.5))
plt.plot(
    d_rx,
    compute_received_power(los_field),
    linestyle="--",
    label="Direct (LoS)",
)
plt.plot(
    d_rx,
    compute_received_power(refl_field),
    linestyle=":",
    label="Reflected only",
)
plt.plot(
    d_rx,
    compute_received_power(total_field),
    label="Total (two-ray interference)",
)
plt.xlabel("Receiver distance from the reflecting wall (m)")
plt.ylabel("Received power (dBW)")
plt.title("Two-ray interference creates multipath fading fringes")
plt.legend()
plt.grid(visible=True, alpha=0.3)
plt.show()

In [ ]:
with dplt.reuse(backend="plotly") as fig:
    wedge.plot(opacity=0.5)
    dplt.draw_markers(
        tx_refl[None, :], labels=["tx"], marker={"color": "red", "size": 5}
    )
    dplt.draw_markers(
        rx_refl[::20], marker={"color": "blue", "size": 3}, name="sampled rx"
    )
    dplt.draw_markers(
        bounce_refl[::20],
        marker={"color": "orange", "size": 3},
        name="specular bounce point",
    )
fig

## 2. Edge Diffraction: The Shadow Region & UTD

Classical geometrical optics predicts abrupt discontinuities at shadow boundaries:
- **Incident Shadow Boundary (ISB)**: where the direct LoS ray becomes occluded.
- **Reflection Shadow Boundary (RSB)**: where the reflected ray can no longer exist.

The Uniform Theory of Diffraction (UTD) introduces diffraction coefficients containing
transition Fresnel integrals $F(X)$ that exactly compensate for these discontinuities,
ensuring the total field remains continuous across all boundaries.

In [ ]:
tx_diff = jnp.array([-10.0, -5.0, 0.0])
angle = jnp.linspace(15.0, 165.0, 600)  # degrees around the shared edge
angle_rad = jnp.radians(angle)
radius = 15.0
rx_diff = radius * jnp.stack(
    [jnp.cos(angle_rad), jnp.sin(angle_rad), jnp.zeros_like(angle_rad)], axis=-1
)
diff_scene = Scene(
    transmitters=tx_diff,
    receivers=rx_diff,
    mesh=wedge,
)
los_paths = diff_scene.trace_paths(order=0)
refl_paths = diff_scene.trace_paths(order=1)
diffraction_paths = single_bounce_paths(
    tx=tx_diff,
    bounce_point=jnp.zeros(3),
    rx=rx_diff,
    object_index=5,  # half-edge 5 = 3 * 1 + 2, the wedge's shared edge
    interaction_type=InteractionType.DIFFRACTION,
)
los_field = compute_received_fields(los_paths, wedge, frequency)[..., 0]
refl_field = compute_received_fields(refl_paths, wedge, frequency).sum(axis=-1)
diff_field = compute_received_fields(diffraction_paths, wedge, frequency)[
    ..., 0
]
total_field = los_field + refl_field + diff_field
# Find the ISB/RSB shadow boundaries: where LoS/reflection existence flips.
los_valid = jnp.any(los_paths.mask, axis=-1)
refl_valid = jnp.any(refl_paths.mask, axis=-1)
isb = angle[jnp.nonzero(jnp.diff(los_valid.astype(int)))[0][0] + 1]
rsb = angle[jnp.nonzero(jnp.diff(refl_valid.astype(int)))[0][0] + 1]
plt.figure(figsize=(9, 5))
plt.plot(angle, compute_received_power(los_field), label="LoS", linewidth=0.8)
plt.plot(
    angle, compute_received_power(refl_field), label="Reflected", linewidth=0.8
)
plt.plot(
    angle, compute_received_power(diff_field), label="Diffracted", linewidth=0.8
)
plt.plot(
    angle,
    compute_received_power(total_field),
    label="Total",
    color="k",
    linewidth=1.2,
)
plt.axvline(float(isb), color="gray", linestyle="--", linewidth=1, label="ISB")
plt.axvline(float(rsb), color="gray", linestyle=":", linewidth=1, label="RSB")
for x, y, label in (
    ((angle[0] + isb) / 2, 0.95, "Shadow\n(diffraction only)"),
    ((isb + rsb) / 2, 0.95, "LoS only"),
    ((rsb + angle[-1]) / 2, 0.08, "LoS + reflection\n(interference)"),
):
    plt.text(
        float(x),
        y,
        label,
        ha="center",
        va="top" if y > 0.5 else "bottom",
        transform=plt.gca().get_xaxis_transform(),
    )
plt.xlabel("Receiver angle around the shared edge (deg)")
plt.ylabel("Received power (dBW)")
plt.title("Diffraction ensures total field continuity across shadow boundaries")
plt.legend(loc="center left")
plt.grid(visible=True, alpha=0.3)
plt.show()

In [ ]:
with dplt.reuse(backend="plotly") as fig:
    wedge.plot(opacity=0.5)
    dplt.draw_markers(
        tx_diff[None, :], labels=["tx"], marker={"color": "red", "size": 5}
    )
    dplt.draw_markers(
        jnp.zeros(3)[None, :],
        labels=["edge point"],
        marker={"color": "green", "size": 5},
    )
    dplt.draw_markers(
        rx_diff[:: rx_diff.shape[0] // 24],
        marker={"color": "blue", "size": 3},
        name="sampled rx",
    )
fig

## 3. Transmission: Attenuation Through a Wall

Transmission accounts for waves passing through finite-thickness dielectric obstacles.
A material requires an explicit `thickness` parameter $d$.
The ray traverses the wall, attenuated by the slab transmission coefficient $T(\theta)$.

In [ ]:
wall = Material(
    name="Concrete", properties=materials["Concrete"].properties, thickness=0.2
)

tx_trans = jnp.array([-5.0, -15.0, 0.0])
y_rx = jnp.linspace(-29.0, -1.0, 300)
rx_trans = jnp.stack(
    [jnp.full_like(y_rx, 5.0), y_rx, jnp.zeros_like(y_rx)], axis=-1
)
bounce_trans = jnp.stack(
    [jnp.zeros_like(y_rx), -7.5 + 0.5 * y_rx, jnp.zeros_like(y_rx)], axis=-1
)  # wall crossing point on plane x=0

transmission_paths = single_bounce_paths(
    tx=tx_trans,
    bounce_point=bounce_trans,
    rx=rx_trans,
    object_index=1,
    interaction_type=InteractionType.TRANSMISSION,
)
transmission_field = compute_received_fields(
    transmission_paths, wedge, frequency, radio_materials={"Concrete": wall}
)

free_space_paths = TracedPaths(
    vertices=jnp.stack(
        [jnp.broadcast_to(tx_trans, rx_trans.shape), rx_trans], axis=-2
    ),
    objects=-jnp.ones((*y_rx.shape, 2), dtype=int),
    mask=jnp.ones_like(y_rx, dtype=bool),
    interaction_types=jnp.zeros((*y_rx.shape, 0), dtype=int),
)
free_space_field = compute_received_fields(free_space_paths, wedge, frequency)

plt.figure(figsize=(8, 4.5))
plt.plot(
    y_rx,
    compute_received_power(free_space_field),
    linestyle="--",
    label="Free space (no wall)",
)
plt.plot(
    y_rx,
    compute_received_power(transmission_field),
    label="Through 20cm concrete wall",
)
plt.xlabel("Receiver position along the wall (m)")
plt.ylabel("Received power (dBW)")
plt.title("Transmission: insertion loss across dielectric slab")
plt.legend()
plt.grid(visible=True, alpha=0.3)
plt.show()

In [ ]:
with dplt.reuse(backend="plotly") as fig:
    wedge.plot(opacity=0.5)
    dplt.draw_markers(
        tx_trans[None, :], labels=["tx"], marker={"color": "red", "size": 5}
    )
    dplt.draw_markers(
        rx_trans[::15], marker={"color": "blue", "size": 3}, name="sampled rx"
    )
    dplt.draw_markers(
        bounce_trans[::15],
        marker={"color": "orange", "size": 3},
        name="wall crossing point",
    )
fig

## 4. Diffuse Scattering: Lambertian Rough Surfaces

Real-world rough surfaces diffuse energy across a broad angular distribution.
Using a material's `scattering_coefficient` $S \in [0, 1]$, scattered fields follow
a Lambertian cosine pattern centered around the surface normal vector $\hat{n}$.

In [ ]:
rough_wall = Material(
    name="Concrete",
    properties=materials["Concrete"].properties,
    scattering_coefficient=0.6,
)

tx_scat = jnp.array([-8.0, -15.0, 8.0])
bounce_scat = jnp.array([0.0, -15.0, 0.0])
radius = 15.0
elevation = jnp.linspace(-85.0, 85.0, 300)  # measured from surface normal
elevation_rad = jnp.radians(elevation)
rx_scat = bounce_scat + radius * jnp.stack(
    [
        -jnp.cos(elevation_rad),
        jnp.zeros_like(elevation_rad),
        jnp.sin(elevation_rad),
    ],
    axis=-1,
)

scattering_paths = single_bounce_paths(
    tx=tx_scat,
    bounce_point=bounce_scat,
    rx=rx_scat,
    object_index=0,
    interaction_type=InteractionType.SCATTERING,
)
scattering_field = compute_received_fields(
    scattering_paths, wedge, frequency, radio_materials={"Concrete": rough_wall}
)

# TX sits at +45 deg from the normal, so the specular direction is at -45 deg.
specular_elevation = -45.0

plt.figure(figsize=(8, 4.5))
plt.plot(
    elevation,
    compute_received_power(scattering_field),
    label="Diffusely scattered power",
)
plt.axvline(
    specular_elevation,
    color="gray",
    linestyle="--",
    label="Specular direction (-45°)",
)
plt.xlabel("Receiver elevation from the surface normal (deg)")
plt.ylabel("Received power (dBW)")
plt.title(
    "Diffuse scattering produces a broad lobe centered on the surface normal"
)
plt.legend()
plt.grid(visible=True, alpha=0.3)
plt.show()

In [ ]:
with dplt.reuse(backend="plotly") as fig:
    wedge.plot(opacity=0.5)
    dplt.draw_markers(
        tx_scat[None, :], labels=["tx"], marker={"color": "red", "size": 5}
    )
    dplt.draw_markers(
        bounce_scat[None, :],
        labels=["bounce point"],
        marker={"color": "green", "size": 5},
    )
    dplt.draw_markers(
        rx_scat[::15], marker={"color": "blue", "size": 3}, name="sampled rx"
    )
fig

## Summary & Custom Solvers

DiffeRT provides a modular EM solver architecture:
- High-level class {class}`TracedFields<differt.em.TracedFields>` wraps field values, delays, operating frequencies, and validity masks.
- {class}`GeometricFieldSolver<differt.em.GeometricFieldSolver>` dispatches each path bounce to its corresponding interaction matrix ({func}`reflection_matrix<differt.em.reflection_matrix>`, {func}`diffraction_matrix<differt.em.diffraction_matrix>`, {func}`scattering_matrix<differt.em.scattering_matrix>`, {func}`transmission_matrix<differt.em.transmission_matrix>`).
- New interaction types (such as `InteractionType.RIS`) can be integrated by subclassing {class}`GeometricFieldSolver<differt.em.GeometricFieldSolver>` or {class}`AbstractFieldSolver<differt.em.AbstractFieldSolver>`.